# 05 — Paired evaluation on labelled SciFact dev claims
This notebook evaluates document retrieval against gold evidence-document IDs. It compares clean top-3, corrupted top-3 and recovered top-3 for the exact same claim IDs. It reports macro-averaged claim-level evidence-document precision/recall/F1, hit@3 and recovery rate. These are retrieval metrics, not final claim-verification label metrics.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
BASE=Path("/content/scifact_study")
manifest=json.loads((BASE/"dataset_manifest.json").read_text())
def read_jsonl(p):
    with open(p,encoding="utf-8") as f: return [json.loads(x) for x in f if x.strip()]
dev=read_jsonl(manifest["dev_path"])
clean=read_jsonl(BASE/"results/clean_rankings.jsonl")
corrupt=read_jsonl(BASE/"results/corrupted_rankings.jsonl")
recovered=read_jsonl(BASE/"results/recovered_rankings.jsonl")
gold={int(r["id"]):set(map(int,r["evidence"].keys())) for r in dev}
sets={
 "clean":{r["claim_id"]:set(r["doc_ids"][:3]) for r in clean},
 "corrupted":{r["claim_id"]:set(r["doc_ids"][:3]) for r in corrupt},
 "recovered":{r["claim_id"]:set(r["doc_ids"][:3]) for r in recovered}
}
claim_ids=set(gold)
assert all(set(x)==claim_ids for x in sets.values()), "Claim IDs differ across conditions"


In [ ]:
def score_condition(pred):
    rows=[]
    for cid in sorted(claim_ids):
        g=gold[cid]; p=pred[cid]
        tp=len(g&p)
        precision=tp/len(p) if p else 0.0
        recall=tp/len(g) if g else (1.0 if not p else 0.0)
        f1=(2*precision*recall/(precision+recall)) if precision+recall else 0.0
        rows.append({"claim_id":cid,"gold_evidence_docs":len(g),"retrieved_docs":len(p),
                     "true_positive_docs":tp,"precision":precision,"recall":recall,
                     "f1":f1,"hit_at_3":int(tp>0)})
    return pd.DataFrame(rows)
per_claim={name:score_condition(pred) for name,pred in sets.items()}
summary=[]
for name,df in per_claim.items():
    summary.append({"condition":name,"n_claims":len(df),
                    "macro_precision":df.precision.mean(),
                    "macro_recall":df.recall.mean(),
                    "macro_f1":df.f1.mean(),
                    "hit_at_3":df.hit_at_3.mean()})
summary_df=pd.DataFrame(summary)
summary_df


In [ ]:
# Paired per-claim recovery accounting: whether recovery fixes a corrupted miss or causes a regression.
c=per_claim["corrupted"].set_index("claim_id")
r=per_claim["recovered"].set_index("claim_id")
clean_df=per_claim["clean"].set_index("claim_id")
fixed=((c.hit_at_3==0)&(r.hit_at_3==1)).sum()
regressed=((c.hit_at_3==1)&(r.hit_at_3==0)).sum()
print("Corrupted misses fixed by recovery:",int(fixed))
print("Previously correct hits lost after recovery:",int(regressed))
print("Recovery rate among corrupted misses:", fixed/max(1,int((c.hit_at_3==0).sum())))


In [ ]:
out=BASE/"results"
summary_df.to_csv(out/"retrieval_metrics_summary.csv",index=False)
for name,df in per_claim.items():
    df.to_csv(out/f"{name}_per_claim_metrics.csv",index=False)
print("Saved summary and per-claim metrics to",out)


## Interpretation guardrails
- These scores evaluate retrieval of annotated evidence abstracts, not the correctness of SUPPORT/CONTRADICT/NOT_ENOUGH_INFO predictions.
- Report the dev split size and confidence intervals in the final paper; use paired resampling by claim for uncertainty.
- Do not call five random seeds five independent datasets. If repeating corruption seeds, aggregate/compare paired claim-level outcomes.